In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
!pip install pyspellchecker
from spellchecker import SpellChecker
!pip install sentence-transformers faiss-cpu numpy --quiet
from sentence_transformers import SentenceTransformer
import faiss
import pickle

In [ ]:
drive.mount('/content/drive')

file_path_interior = '/content/drive/My Drive/dataset_extracted_all_record.csv'
df_interior_db_ = pd.read_csv(file_path_interior)

df_with_ROS=df_interior_db_.copy()

# Display first few rows
print("First 5 rows of the dataset:")
display(df_interior_db_)

# Show dataset shape
print(f"\nDataset contains {df_interior_db_.shape[0]} rows and {df_interior_db_.shape[1]} columns.")

# Show column names
print("\nColumns in the dataset:")
print(df_interior_db_.columns.tolist())

# Show dataset info
print("\nDataset Info:")
df_interior_db_.info()

# Show basic statistics
print("\nDataset Statistics:")
display(df_interior_db_.describe(include="all"))

Clean the Description Col

In [ ]:


# Initialize spell checker
spell = SpellChecker()
def clean_description(text):
    # Lowercase
    text = text.lower()

    # Remove non-letter characters (keep spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Split into words
    words = text.split()

    # Remove repeated words (while preserving order)
    seen = set()
    unique_words = []
    for word in words:
        if word not in seen:
            seen.add(word)
            unique_words.append(word)

    # Correct spelling
    corrected = [spell.correction(word) if word not in spell else word for word in unique_words]

    return ' '.join(corrected)


In [ ]:
import re

# Apply cleaning to description column
df_interior_db_['cleaned_description'] = df_interior_db_['description'].astype(str).apply(clean_description)

df_with_ROS['cleaned_description'] = df_with_ROS['description'].astype(str).apply(clean_description)

# Preview cleaned version
display(df_interior_db_[['description', 'cleaned_description']].head(5))
display(df_with_ROS[['description', 'cleaned_description']].head(5))


In [ ]:

import string
import nltk
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
import tensorflow_hub as hub
import torch
from transformers import BertTokenizer, BertModel
from google.colab import drive
import pandas as pd
# Ensure necessary NLTK resources are downloaded
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

nltk.download('punkt')
nltk.download('stopwords')
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()  # Convert to lowercase
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove punctuation
        tokens = word_tokenize(text)  # Tokenization
        tokens = [word for word in tokens if word not in stopwords.words('english')]  # Remove stopwords
        return ' '.join(tokens)
    return text

df_interior_db_['description'] = df_interior_db_['description'].apply(clean_text)


# Random Over Sampling df_with_ROS

In [ ]:
# Class distribution before balancing
plt.figure(figsize=(10, 5))
sns.countplot(x=df_with_ROS['predicted_style'], palette='viridis')
plt.title("Class Distribution Before Balancing")
plt.xlabel("Predicted Style")
plt.ylabel("Count")
plt.show()

# Handle Class Imbalance in 'predicted_style' Using Random Over-Sampling (ROS)
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(df_with_ROS.drop(columns=['predicted_style']), df_with_ROS['predicted_style'])
df_with_ROS = pd.concat([X_resampled, y_resampled], axis=1)

# Class distribution after balancing
plt.figure(figsize=(10, 5))
sns.countplot(x=df_with_ROS['predicted_style'], palette='viridis')  # Updated to use df_interior_db
plt.title("Class Distribution After Balancing")
plt.xlabel("Predicted Style")
plt.ylabel("Count")
plt.show()


print("Class imbalance handled and descriptions cleaned.")

In [ ]:
print("Dataset Without ROS")

df_interior_db_.shape
df_interior_db_.describe()

In [ ]:
print("Dataset With ROS")
df_with_ROS.shape
df_with_ROS.describe()


Prediction without ROS

In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

# Your strict style rules
STYLE_RULES = {
    'ISTP': ['Vintage', 'Modern'],
    'ISTJ': ['Modern', 'Bohemian', 'Vintage'],
    'ISFP': ['Vintage'],
    'ISFJ': ['Minimalist', 'Vintage', 'Bohemian'],
    'INTP': ['Modern', 'Industrial', 'Vintage'],
    'INTJ': ['Vintage', 'Industrial', 'Bohemian', 'Modern'],
    'INFP': ['Modern', 'Vintage', 'Minimalist'],
    'INFJ': ['Industrial', 'Vintage', 'Modern'],
    'ESTP': ['Modern', 'Vintage'],
    'ESTJ': ['Modern', 'Vintage', 'Bohemian'],
    'ESFP': ['Bohemian', 'Modern'],
    'ESFJ': ['Vintage'],
    'ENTP': ['Modern', 'Vintage'],
    'ENTJ': ['Vintage', 'Bohemian', 'Modern'],
    'ENFP': ['Vintage', 'Bohemian', 'Industrial'],
    'ENFJ': ['Modern', 'Vintage']
}

def create_correlation_matrix():
    """Create personality correlation matrix based on style rules"""
    # Convert rules to DataFrame
    df_rules = pd.DataFrame([
        {'personality_type': pt, 'styles': styles}
        for pt, styles in STYLE_RULES.items()
    ])

    # One-hot encode styles
    mlb = MultiLabelBinarizer()
    style_encoded = mlb.fit_transform(df_rules['styles'])

    # Calculate cosine similarity
    corr_matrix = pd.DataFrame(
        cosine_similarity(style_encoded),
        columns=df_rules['personality_type'],
        index=df_rules['personality_type']
    )

    return corr_matrix

def fetch_matching_designs(personality_type, df_designs):
    """Fetch designs matching personality type's styles"""
    if personality_type not in STYLE_RULES:
        raise ValueError(f"Invalid personality type. Must be one of: {list(STYLE_RULES.keys())}")

    target_styles = STYLE_RULES[personality_type]

    # Filter designs where predicted_style matches any target style
    mask = df_designs['predicted_style'].apply(
        lambda x: any(style.lower() in x.lower() for style in target_styles)
    )

    return df_designs[mask].copy()

# Example usage
if __name__ == "__main__":
    # Assuming df_interior_db is already loaded with your columns:
    # folder_name, image_name, dominant_color, palette, description, predicted_style

    # Create correlation matrix
    corr_matrix = create_correlation_matrix()
    print("Personality Style Correlation Matrix:")
    print(corr_matrix)

    # Get user input
    #while True:
    user_input = input("\nEnter personality type (e.g., ISTP, ENFJ) or 'quit': ").strip().upper()

     #   if user_input == 'QUIT':
      #      break

    try:
            # 1. Show similar personalities
            print(f"\nMost similar personalities to {user_input}:")
            similar = corr_matrix[user_input].sort_values(ascending=False)[1:4]
            print(similar.to_string())
            # 2. Fetch matching designs
            matching_designs = fetch_matching_designs(user_input, df_interior_db_)

# ✅ Add personality type to each row
            matching_designs["personality_type"] = user_input

            print(f"\nFound {len(matching_designs)} matching designs:")
            print("="*80)
            print(matching_designs[['folder_name', 'image_name', 'predicted_style', 'description','cleaned_description', 'personality_type']].head())
            print("="*80)

# ✅ Save to CSV with type info
            matching_designs.to_csv(f"{user_input}_designs.csv", index=False)


            # Optional: save full results

    except ValueError as e:
            print(f"Error: {e}")

# Prediction with Random over sampling for ISTP

In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

# Your strict style rules
STYLE_RULES = {
    'ISTP': ['Vintage', 'Modern'],
    'ISTJ': ['Modern', 'Bohemian', 'Vintage'],
    'ISFP': ['Vintage'],
    'ISFJ': ['Minimalist', 'Vintage', 'Bohemian'],
    'INTP': ['Modern', 'Industrial', 'Vintage'],
    'INTJ': ['Vintage', 'Industrial', 'Bohemian', 'Modern'],
    'INFP': ['Modern', 'Vintage', 'Minimalist'],
    'INFJ': ['Industrial', 'Vintage', 'Modern'],
    'ESTP': ['Modern', 'Vintage'],
    'ESTJ': ['Modern', 'Vintage', 'Bohemian'],
    'ESFP': ['Bohemian', 'Modern'],
    'ESFJ': ['Vintage'],
    'ENTP': ['Modern', 'Vintage'],
    'ENTJ': ['Vintage', 'Bohemian', 'Modern'],
    'ENFP': ['Vintage', 'Bohemian', 'Industrial'],
    'ENFJ': ['Modern', 'Vintage']
}

def create_correlation_matrix():
    """Create personality correlation matrix based on style rules"""
    # Convert rules to DataFrame
    df_rules = pd.DataFrame([
        {'personality_type': pt, 'styles': styles}
        for pt, styles in STYLE_RULES.items()
    ])

    # One-hot encode styles
    mlb = MultiLabelBinarizer()
    style_encoded = mlb.fit_transform(df_rules['styles'])

    # Calculate cosine similarity
    corr_matrix = pd.DataFrame(
        cosine_similarity(style_encoded),
        columns=df_rules['personality_type'],
        index=df_rules['personality_type']
    )

    return corr_matrix

def fetch_matching_designs(personality_type, df_designs):
    """Fetch designs matching personality type's styles"""
    if personality_type not in STYLE_RULES:
        raise ValueError(f"Invalid personality type. Must be one of: {list(STYLE_RULES.keys())}")

    target_styles = STYLE_RULES[personality_type]

    # Filter designs where predicted_style matches any target style
    mask = df_designs['predicted_style'].apply(
        lambda x: any(style.lower() in x.lower() for style in target_styles)
    )

    return df_designs[mask].copy()

# Example usage
if __name__ == "__main__":
    # Assuming df_interior_db is already loaded with your columns:
    # folder_name, image_name, dominant_color, palette, description, predicted_style

    # Create correlation matrix
    corr_matrix = create_correlation_matrix()
    print("Personality Style Correlation Matrix:")
    print(corr_matrix)

    # Get user input
    #while True:
    user_input = input("\nEnter personality type (e.g., ISTP, ENFJ) or 'quit': ").strip().upper()

     #   if user_input == 'QUIT':
      #      break

    try:
            # 1. Show similar personalities
            print(f"\nMost similar personalities to {user_input}:")
            similar = corr_matrix[user_input].sort_values(ascending=False)[1:4]
            print(similar.to_string())
            # 2. Fetch matching designs
            matching_designs = fetch_matching_designs(user_input, df_with_ROS)

# ✅ Add personality type to each row
            matching_designs["personality_type"] = user_input

            print(f"\nFound {len(matching_designs)} matching designs:")
            print("="*80)
            print(matching_designs[['folder_name', 'image_name', 'predicted_style', 'description','cleaned_description', 'personality_type']].head())
            print("="*80)

# ✅ Save to CSV with type info
            matching_designs.to_csv(f"{user_input}_designs_ROS.csv", index=False)


            # Optional: save full results

    except ValueError as e:
            print(f"Error: {e}")

# Coorelation of personality style pref with interior design predicted style

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Your strict style rules
STYLE_RULES = {
    'ISTP': ['Vintage', 'Modern'],
    'ISTJ': ['Modern', 'Bohemian', 'Vintage'],
    'ISFP': ['Vintage'],
    'ISFJ': ['Minimalist', 'Vintage', 'Bohemian'],
    'INTP': ['Modern', 'Industrial', 'Vintage'],
    'INTJ': ['Vintage', 'Industrial', 'Bohemian', 'Modern'],
    'INFP': ['Modern', 'Vintage', 'Minimalist'],
    'INFJ': ['Industrial', 'Vintage', 'Modern'],
    'ESTP': ['Modern', 'Vintage'],
    'ESTJ': ['Modern', 'Vintage', 'Bohemian'],
    'ESFP': ['Bohemian', 'Modern'],
    'ESFJ': ['Vintage'],
    'ENTP': ['Modern', 'Vintage'],
    'ENTJ': ['Vintage', 'Bohemian', 'Modern'],
    'ENFP': ['Vintage', 'Bohemian', 'Industrial'],
    'ENFJ': ['Modern', 'Vintage']
}

def create_full_style_heatmap(designs_db):
    """
    Creates a heatmap comparing ALL personality types vs ALL styles in the DB.
    Uses your exact STYLE_RULES for comparison.
    """
    try:
        # --- 1. Prepare Data ---
        # Get ALL unique styles from DB (lowercase for consistency)
        all_db_styles = sorted(designs_db['predicted_style'].str.lower().unique())

        # Get ALL personality types from your rules
        personality_types = sorted(STYLE_RULES.keys())

        # --- 2. Initialize Correlation Matrix ---
        corr_matrix = pd.DataFrame(
            index=personality_types,
            columns=all_db_styles,
            data=0.0  # Default: no correlation
        )

        # --- 3. Calculate Correlations ---
        for personality in personality_types:
            preferred_styles = [s.lower() for s in STYLE_RULES[personality]]

            for db_style in all_db_styles:
                # Case 1: Exact match (1.0)
                if db_style in preferred_styles:
                    corr_matrix.loc[personality, db_style] = 1.0
                # Case 2: Partial match (0.5)
                elif any(db_style in s or s in db_style for s in preferred_styles):
                    corr_matrix.loc[personality, db_style] = 0.5
                # Else: 0.0 (no correlation)

        # --- 4. Plotting ---
        plt.figure(figsize=(16, 10))

        # Custom colormap: white (0.0) -> yellow (0.5) -> green (1.0)
        cmap = LinearSegmentedColormap.from_list("custom", ["white", "gold", "green"])

        sns.heatmap(
            corr_matrix,
            annot=True,
            fmt=".1f",
            cmap=cmap,
            vmin=0,
            vmax=1,
            linewidths=0.5,
            cbar_kws={'label': 'Correlation Strength'}
        )

        plt.title(
            "Personality-Style Relationships\n"
            "Green = Exact Match (1.0) | Yellow = Partial Match (0.5) | White = No Correlation (0.0)",
            pad=20,
            fontsize=14
        )
        plt.xlabel("Interior Design DB Styles")
        plt.ylabel("MBTI Personality Types")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

        return corr_matrix

    except Exception as e:
        print(f"Error creating heatmap: {str(e)}")
        return None

# Example usage
if __name__ == "__main__":
    # Load your interior design dataset
    # df_interior_db = pd.read_csv("your_designs.csv")
    # For demonstration, creating a mock dataset:


    # Generate the heatmap
    correlation_data = create_full_style_heatmap(df_interior_db_)

    if correlation_data is not None:
        print("\nCorrelation Matrix Summary:")
        print(correlation_data)

# Hybrid Retrieval (Optimized for Design Data) Technique: "Multi-Modal Embeddings"

In [ ]:
import ast
import matplotlib.colors as mcolors

# Parse the palette string into a list of RGB tuples
def parse_palette(palette_str):
    try:
        return ast.literal_eval(palette_str)
    except:
        return []

# Convert RGB to hex color string
def rgb_to_color_name(rgb_tuple):
    try:
        norm_rgb = tuple(x / 255.0 for x in rgb_tuple)
        return mcolors.to_hex(norm_rgb)
    except:
        return "unknown"

# Apply processing to the DataFrame
df_type['palette_names'] = df_type['palette'].apply(
    lambda x: ' '.join([rgb_to_color_name(rgb) for rgb in parse_palette(x)])
)

# Show example output
print(df_type[['palette', 'palette_names']].head())


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle
import faiss

# Initialize model
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# Combine all relevant features into one embedding string
embedding_inputs = [
    f"Personality: {row['personality_type']} | Color: {row['dominant_color']} | Style: {row['predicted_style']} | Palette: {row['palette_names']} | Description: {row['cleaned_description']}"
    for _, row in df_type.iterrows()
]

# Generate embeddings
embeddings = model.encode(embedding_inputs)
embeddings = np.array(embeddings).astype('float32')

# Save embeddings and context
with open('design_embeddings.pkl', 'wb') as f:
    pickle.dump({
        'embeddings': embeddings,
        'design_contexts': embedding_inputs  # full string input used
    }, f)

# Save FAISS index
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)
faiss.write_index(faiss_index, 'design_index.faiss')

print("✅ Embeddings generated and saved with personality type included.")


In [ ]:
# Load embeddings
with open('design_embeddings.pkl', 'rb') as f:
    data = pickle.load(f)
    embeddings = data['embeddings']
    designs = data['design_contexts']

# Load FAISS index
index = faiss.read_index('design_index.faiss')

# Verify loading
print(f"Loaded {len(designs)} designs with {embeddings.shape[1]}-dim embeddings")

In [ ]:
# Create FAISS index and add embeddings
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)  # <-- THIS is crucial

# Save index
faiss.write_index(index, 'design_index.faiss')


In [ ]:
def retrieve_designs(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)
    retrieved_contexts = "\n\n".join([designs[i] for i in indices[0]])
    return retrieved_contexts, query_embedding


In [ ]:
for i, row in df_type.iterrows():
    query_text = f"Personality: {row['personality_type']} | Style: {row['predicted_style']} | Palette: {row['palette_names']} | Description: {row['cleaned_description']}"

    print(f"\n Generated Query for Row {i}:\n{query_text}\n")

    retrieved_contexts, query_embedding = retrieve_designs(query_text)



In [ ]:
def evaluate_retrieval(query, relevant_indices):
    retrieved = retrieve_designs(query)
    precision = len(set(retrieved) & set(relevant_indices)) / len(retrieved)
    return precision

In [ ]:
from sklearn.cluster import KMeans

# Choose a suitable number of clusters (e.g., 10–30 depending on data size)
num_clusters = 20
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(embeddings)

design_clusters = {i: cluster_labels[i] for i in range(len(designs))}


def generate_cluster_filtered_prompts(k=3):
    prompts = []
    for i, context in enumerate(designs):
        current_cluster = design_clusters[i]
        query_embedding = np.expand_dims(embeddings[i], axis=0)

        distances, indices = index.search(query_embedding, 50)  # Get more candidates
        filtered_indices = [
            idx for idx in indices[0]
            if design_clusters[idx] != current_cluster and idx != i
        ][:k]

        similar_contexts = [designs[j] for j in filtered_indices]
        prompt = f"""Design: "{context}"\n\nTop {k} diverse design inspirations:\n""" + \
                 "\n\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)])

        prompts.append(prompt)

    return prompts
cluster_prompts = generate_cluster_filtered_prompts(k=3)
for p in cluster_prompts:
    print(p)
    print("=" * 50)


Evalute Prompts

In [ ]:
def evaluate_all_prompts(all_prompts, k=3):
    """Evaluates retrieval quality for all generated prompts"""
    results = []

    for i, prompt in enumerate(all_prompts[:5]):  # Evaluate first 5 for demo
        # Extract the query (original design)
        query = designs[i]

        # Get ground truth - indices of similar designs found during prompt generation
        query_embedding = np.expand_dims(embeddings[i], axis=0)
        _, indices = index.search(query_embedding, k+1)
        relevant_indices = [idx for idx in indices[0] if idx != i][:k]

        # Evaluate
        precision = evaluate_retrieval(query, relevant_indices)

        results.append({
            'design_idx': i,
            'query': query[:50] + "...",  # First 50 chars
            'precision': precision,
            'relevant_indices': relevant_indices
        })

    return results

In [ ]:
def generate_automatic_prompts_with_scores(k=3):
    prompts = []
    for i, context in enumerate(designs):
        query_embedding = np.expand_dims(embeddings[i], axis=0)
        distances, indices = index.search(query_embedding, k+1)  # +1 to skip self

        result = []
        for j, idx in enumerate(indices[0]):
            if idx != i and len(result) < k:
                score = distances[0][j]
                result.append((score, designs[idx]))

        prompt = f"""Design: "{context}"

Top {k} similar design inspirations (with FAISS L2 distances):
""" + "\n\n".join([f"{j+1}. (Score: {score:.4f}) {sc}" for j, (score, sc) in enumerate(result)])

        prompts.append(prompt)

    return prompts


In [ ]:
pip install openai


In [ ]:
from openai import OpenAI
import time

# Initialize OpenAI client
client = OpenAI(api_key="My_key")  # Replace with your key

# Function to get GPT responses for your auto-generated prompts
def generate_responses_from_prompts(prompts, model="gpt-4o-mini", temperature=0.7, max_tokens=200):
    responses = []

    for i, prompt in enumerate(prompts):
        print(f"Sending prompt {i+1}/{len(prompts)}...")

        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a helpful interior design assistant."},
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )
            response_text = completion.choices[0].message.content.strip()
            responses.append((prompt, response_text))
        except Exception as e:
            print(f"Error with prompt {i}: {e}")
            responses.append((prompt, "ERROR"))

        time.sleep(1)  # Avoid rate limits

    return responses


In [ ]:
# Use your generated prompts (e.g., from generate_automatic_prompts())
test_prompts = all_prompts[:5]  # Only send first 5 to avoid cost/rate issues

results = generate_responses_from_prompts(test_prompts)

# Print results
for i, (prompt, response) in enumerate(results):
    print(f"\n--- PROMPT {i+1} ---\n{prompt}")
    print(f"\n--- GPT-4o-mini RESPONSE ---\n{response}")
    print("="*80)


# Generate Recommendations with balance and unbalance dataset (RAG based approach)


# Applied RAG on  Random over sampling and included Description

In [ ]:


df_type = pd.read_csv(f"ISTP_designs_ROS.csv")


# --- Optional: Show column names to verify structure ---
print("✅ Columns in loaded CSV:", df_type.columns.tolist())


# --- Construct context using all relevant features ---
df_type["design_context_without_desc"] = df_type.apply(
    lambda row: f"""
    Personality Type: {row['personality_type']}
    Design Details:
    - Dominant Color: {row['dominant_color']}
    - Color Palette: {row['palette']}
    - style: {row['predicted_style']}
    - description: {row['cleaned_description']}

    """,
    axis=1
)

# --- Convert context column to list for downstream processing ---
designs = df_type["design_context_without_desc"].tolist()

# --- Display first few design contexts ---
print("\n✅ Sample design contexts:")
for d in designs[:3]:
    print(d)
    print("=" * 50)

In [ ]:
import ast
import matplotlib.colors as mcolors
from scipy.spatial import KDTree
import numpy as np

# Create a KDTree for fast nearest neighbor search
css3_db = mcolors.CSS4_COLORS  # You can also use CSS4_COLORS for a richer set
names = list(css3_db.keys())
rgb_values = [mcolors.to_rgb(css3_db[name]) for name in names]
kdtree = KDTree(rgb_values)

# Parse the palette string into a list of RGB tuples
def parse_palette(palette_str):
    try:
        return ast.literal_eval(palette_str)
    except:
        return []

# Convert RGB tuple to nearest color name
def rgb_to_color_name(rgb_tuple):
    try:
        norm_rgb = tuple(x / 255.0 for x in rgb_tuple)
        _, idx = kdtree.query(norm_rgb)
        return names[idx]
    except:
        return "unknown"

# Apply processing to the DataFrame
df_type['palette_names'] = df_type['palette'].apply(
    lambda x: ', '.join([rgb_to_color_name(rgb) for rgb in parse_palette(x)])
)

# Show example output
print(df_type[['palette', 'palette_names']].head())


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle
import faiss

# Initialize model
model = SentenceTransformer('paraphrase-MiniLM-L12-v2')

# Combine all relevant features into one embedding string
embedding_inputs = [
    f"Personality: {row['personality_type']} | Color: {row['dominant_color']} | Style: {row['predicted_style']} | Palette: {row['palette_names']} | description: {row['cleaned_description']}"
    for _, row in df_type.iterrows()
]

# Generate embeddings
embeddings = model.encode(embedding_inputs)
embeddings = np.array(embeddings).astype('float32')

# Save embeddings and context
with open('design_embeddings_with_ROS.pkl', 'wb') as f:
    pickle.dump({
        'embeddings': embeddings,
        'design_contexts': embedding_inputs  # full string input used
    }, f)

# Save FAISS index
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)
faiss.write_index(faiss_index, 'design_index_with_ROS.faiss')

print("✅ Embeddings generated and saved with personality type included.")


In [ ]:
# Load embeddings
with open('design_embeddings_with_ROS.pkl', 'rb') as f:
    data = pickle.load(f)
    embeddings = data['embeddings']
    designs = data['design_contexts']

# Load FAISS index
index = faiss.read_index('design_index_with_ROS.faiss')

# Verify loading
print(f"Loaded {len(designs)} designs with {embeddings.shape[1]}-dim embeddings")

In [ ]:
def retrieve_designs(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)
    retrieved_contexts = "\n\n".join([designs[i] for i in indices[0]])
    return retrieved_contexts, query_embedding

In [ ]:
# Create FAISS index and add embeddings
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)  # <-- THIS is crucial

# Save index
faiss.write_index(index, 'design_index_with_ROS_istp.faiss')


# TEST Cases Prompt generation

In [ ]:
def generate_design_prompts_istp(k=3):
    prompts = []

    for i, context in enumerate(designs):
        query_embedding = np.expand_dims(embeddings[i], axis=0)  # Get the embedding of the current design context
        distances, indices = faiss_index.search(query_embedding, k + 1)  # Search for the most similar designs
        similar_indices = [idx for idx in indices[0] if idx != i][:k]  # Exclude self and take the top k similar indices

        similar_contexts = [designs[j] for j in similar_indices]  # Get similar contexts based on the indices

        # Refined and concise prompt for better relevance and creativity
        prompt = f"""You are a visionary interior designer with a deep understanding of personality-driven design. Your task is to create a space that not only represents the user’s personality but also nurtures their emotional needs.

**Primary Design Inspiration:**
{context}

**Related Inspirations:**
""" + "\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)])

        prompt += f"""
Design this space thoughtfully, considering the user’s emotional connection to the environment:
- Choose a **primary color** that captures the user’s mood and enhances their personality.
- Select **furniture** that aligns with their emotional needs, comfort, and expression.
- Pick **lighting** that either energizes or calms, depending on their preference.

Finally, describe the **overall emotional vibe** of the space and explain how the elements of design (color, furniture, and lighting) seamlessly come together to create a cohesive and emotionally fulfilling environment.
"""

        prompts.append(prompt)

    return prompts

# --- Now generate the design prompts ---
design_prompts = generate_design_prompts_istp(k=3)

# --- Preview the first few prompts ---
for p in design_prompts[:1]:
    print(p)
    print("=" * 80)


In [ ]:
def generate_design_prompts_istps(k=3):
    prompts = []
    top_k_references_all = []  # store top-k references per prompt

    for i, context in enumerate(designs):
        query_embedding = np.expand_dims(embeddings[i], axis=0)
        distances, indices = faiss_index.search(query_embedding, k + 1)
        similar_indices = [idx for idx in indices[0] if idx != i][:k]

        similar_contexts = [designs[j] for j in similar_indices]

        prompt = f"""You are a visionary interior designer with a deep understanding of personality-driven design. Your task is to create a space that not only represents the user’s personality but also nurtures their emotional needs.

**Primary Design Inspiration:**
{context}

**Related Inspirations:**
""" + "\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)])

        prompt += f"""
Design this space thoughtfully, considering the user’s emotional connection to the environment:
- Choose a **primary color** that captures the user’s mood and enhances their personality.
- Select **furniture** that aligns with their emotional needs, comfort, and expression.
- Pick **lighting** that either energizes or calms, depending on their preference.

Finally, describe the **overall emotional vibe** of the space and explain how the elements of design (color, furniture, and lighting) seamlessly come together to create a cohesive and emotionally fulfilling environment.
"""

        prompts.append(prompt)
        top_k_references_all.append(similar_contexts)  # save top-k references

    return prompts, top_k_references_all

design_prompts, top_k_references_all = generate_design_prompts_istps(k=3)

# Preview
for p, refs in zip(design_prompts[:1], top_k_references_all[:1]):
    print("Prompt:\n", p)
    print("\nTop-k References:\n", refs)
    print("="*80)



In [ ]:
import requests
import time

api_key="my_key";
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
#changed the api and version from llama3 70 b to llama 3.3 70b versatile
def generate_llama3_recommendations(prompts, model="llama-3.3-70b-versatile", temperature=0.5):
    responses = []
    for i, prompt in enumerate(prompts):
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a brilliant, emotionally intuitive interior designer. Make your recommendations creative,concise, expressive,complete and glitterified."},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "max_tokens": 768
        }

        try:
            res = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)

            # Check for errors
            if res.status_code != 200:
                print(f"[❌] Prompt {i+1} failed: HTTP {res.status_code} → {res.text}")
                responses.append("")
                continue

            data = res.json()
            output = data["choices"][0]["message"]["content"]
            print(f"[✅] Prompt {i+1} completed.")
            responses.append(output)

        except requests.exceptions.RequestException as e:
            print(f"[❌] Network error for prompt {i+1}: {e}")
            responses.append("")
        except Exception as e:
            print(f"[❌] Unexpected error for prompt {i+1}: {e}")
            responses.append("")

        time.sleep(0.5)  # avoid rate limits
    return responses



In [ ]:
# Assuming you've already created semantic_prompts
llama3_outputs = generate_llama3_recommendations(design_prompts[:1])  # Do first
display(llama3_outputs)

# Evalution of test case 1 ISTP

In [ ]:
!pip install rouge_score
!pip install bert-score


In [ ]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
import numpy as np
from nltk.util import ngrams

# --- Load SBERT ---
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

# --- Text normalization ---
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- Semantic P-BLEU (reference-free, input-based) ---
def semantic_p_bleu(hypo_text, topk_refs, n=3, sim_threshold=0.5):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    scores = []
    for ngram_size in range(1, n+1):
        hypo_ngrams = list(ngrams(hypo_tokens, ngram_size))
        hypo_str_ngrams = [" ".join(ng) for ng in hypo_ngrams]
        hypo_emb = model_sbert.encode(hypo_str_ngrams, convert_to_tensor=True)

        max_match_scores = []
        for ref_text in topk_refs:
            ref_tokens = normalize_text(ref_text).split()
            ref_ngrams = list(ngrams(ref_tokens, ngram_size))
            ref_str_ngrams = [" ".join(ng) for ng in ref_ngrams]
            if not ref_str_ngrams:
                continue
            ref_emb = model_sbert.encode(ref_str_ngrams, convert_to_tensor=True)

            cos_sim = util.cos_sim(hypo_emb, ref_emb)
            max_per_hypo = cos_sim.max(dim=1).values
            max_per_hypo = (max_per_hypo >= sim_threshold).float()
            max_match_scores.append(max_per_hypo.sum().item() / len(hypo_ngrams))
        scores.append(max(max_match_scores) if max_match_scores else 0)
    return np.mean(scores)

# --- Semantic ROUGE-L (reference-free) ---
def semantic_rouge_l(hypo_text, topk_refs, sim_threshold=0.6):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    max_score = 0.0
    hypo_emb = model_sbert.encode(hypo_tokens, convert_to_tensor=True)

    for ref_text in topk_refs:
        ref_tokens = normalize_text(ref_text).split()
        if not ref_tokens:
            continue
        ref_emb = model_sbert.encode(ref_tokens, convert_to_tensor=True)
        cos_sim = util.cos_sim(hypo_emb, ref_emb)
        binary_matches = (cos_sim >= sim_threshold).float()
        precision = binary_matches.max(dim=1).values.mean().item()
        recall = binary_matches.max(dim=0).values.mean().item()
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        max_score = max(max_score, f1)
    return max_score

# --- Evaluation ---
results = []

for i, hypo_text in enumerate(llama3_outputs):
    topk_refs = top_k_references_all[i]  # exact 3 similar inputs used in prompt

    p_bleu = semantic_p_bleu(hypo_text, topk_refs)
    rouge_l = semantic_rouge_l(hypo_text, topk_refs)

    # Compute BERTScore separately

    results.append({
        "P-BLEU": round(p_bleu,3),
        "ROUGE-L": round(rouge_l,3),
    })

df = pd.DataFrame(results)
display(df)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score

# --- Load SBERT model ---
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

# --- Initialize results list ---
results = []

# Loop through all generated recommendations
for i in range(len(llama3_outputs)):
    generated = llama3_outputs[i]
    refs = top_k_references_all[i]  # list of top-k reference strings for this prompt

    # --- BERTScore (precision, recall, F1) ---
    # Pick best reference based on max SBERT similarity for BERTScore
    best_ref = max(refs, key=lambda r: util.cos_sim(
        model_sbert.encode(r, convert_to_tensor=True),
        model_sbert.encode(generated, convert_to_tensor=True)
    ).item())
    P, R, F1 = bert_score([generated], [best_ref], lang="en")
    P, R, F1 = float(P[0]), float(R[0]), float(F1[0])

    # --- SBERT semantic alignment (max cosine similarity among top-k refs) ---
    emb_refs = model_sbert.encode(refs, convert_to_tensor=True)
    emb_gen = model_sbert.encode(generated, convert_to_tensor=True)
    cos_scores = util.cos_sim(emb_gen, emb_refs)
    sbert_alignment = cos_scores.max().item()

    # --- Save results ---
    results.append({
        "BERTScore_P": round(P, 3),
        "BERTScore_R": round(R, 3),
        "BERTScore_F1": round(F1, 3),
        "SBERT_Alignment": round(sbert_alignment, 3)
    })

# --- Create DataFrame ---
df_metrics = pd.DataFrame(results)
display(df_metrics)

# ==========================================================
# --- Graphical Representation ---
# ==========================================================
plt.figure(figsize=(12,6))
plt.plot(df_metrics.index, df_metrics["BERTScore_F1"], marker='o', label="BERTScore F1")
plt.plot(df_metrics.index, df_metrics["SBERT_Alignment"], marker='s', label="SBERT Alignment (Cosine)")
plt.title("Evaluation of Generated Recommendations")
plt.xlabel("Prompt Index")
plt.ylabel("Score")
plt.ylim(0, 1.0)
plt.grid(True)
plt.legend()
plt.show()

# Optional: side-by-side bar chart
df_metrics[["BERTScore_F1","SBERT_Alignment"]].plot(
    kind="bar", figsize=(12,6), rot=0, title="BERTScore vs SBERT Alignment"
)
plt.ylabel("Score")
plt.show()


# Test Case For ENFP

RAG Applied

In [ ]:


df_type = pd.read_csv(f"ENFP_designs_ROS.csv")


# --- Optional: Show column names to verify structure ---
print("✅ Columns in loaded CSV:", df_type.columns.tolist())


# --- Construct context using all relevant features ---
df_type["design_context_without_desc"] = df_type.apply(
    lambda row: f"""
    Personality Type: {row['personality_type']}
    Design Details:
    - Dominant Color: {row['dominant_color']}
    - Color Palette: {row['palette']}
    - style: {row['predicted_style']}
    - description: {row['cleaned_description']}

    """,
    axis=1
)

# --- Convert context column to list for downstream processing ---
designs = df_type["design_context_without_desc"].tolist()

# --- Display first few design contexts ---
print("\n✅ Sample design contexts:")
for d in designs[:3]:
    print(d)
    print("=" * 50)

In [ ]:
import ast
import matplotlib.colors as mcolors
from scipy.spatial import KDTree
import numpy as np

# Create a KDTree for fast nearest neighbor search
css3_db = mcolors.CSS4_COLORS  # You can also use CSS4_COLORS for a richer set
names = list(css3_db.keys())
rgb_values = [mcolors.to_rgb(css3_db[name]) for name in names]
kdtree = KDTree(rgb_values)

# Parse the palette string into a list of RGB tuples
def parse_palette(palette_str):
    try:
        return ast.literal_eval(palette_str)
    except:
        return []

# Convert RGB tuple to nearest color name
def rgb_to_color_name(rgb_tuple):
    try:
        norm_rgb = tuple(x / 255.0 for x in rgb_tuple)
        _, idx = kdtree.query(norm_rgb)
        return names[idx]
    except:
        return "unknown"

# Apply processing to the DataFrame
df_type['palette_names'] = df_type['palette'].apply(
    lambda x: ', '.join([rgb_to_color_name(rgb) for rgb in parse_palette(x)])
)

# Show example output
print(df_type[['palette', 'palette_names']].head())


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle
import faiss

# Initialize model
model = SentenceTransformer('paraphrase-MiniLM-L12-v2')

# Combine all relevant features into one embedding string
embedding_inputs = [
    f"Personality: {row['personality_type']} | Color: {row['dominant_color']} | Style: {row['predicted_style']} | Palette: {row['palette_names']} | description: {row['cleaned_description']}"
    for _, row in df_type.iterrows()
]

display(embedding_inputs)
# Generate embeddings
embeddings = model.encode(embedding_inputs)
embeddings = np.array(embeddings).astype('float32')

# Save embeddings and context
with open('design_embeddings_with_ROS_ENFP.pkl', 'wb') as f:
    pickle.dump({
        'embeddings': embeddings,
        'design_contexts': embedding_inputs  # full string input used
    }, f)

# Save FAISS index
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)
faiss.write_index(faiss_index, 'design_index_with_ROS_ENFP.faiss')

print("✅ Embeddings generated and saved with personality type included.")


In [ ]:
# Load embeddings
import pickle
import faiss

with open('design_embeddings_with_ROS_ENFP.pkl', 'rb') as f:
    data = pickle.load(f)
    embeddings = data['embeddings']
    designs = data['design_contexts']

# Load FAISS index
index = faiss.read_index('design_index_with_ROS_ENFP.faiss')

# Verify loading
print(f"Loaded {len(designs)} designs with {embeddings.shape[1]}-dim embeddings")
# Create FAISS index and add embeddings
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# Save index
faiss.write_index(index, 'design_index_with_ROS_ENFP.faiss')


In [ ]:
def retrieve_designs(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)
    retrieved_contexts = "\n\n".join([designs[i] for i in indices[0]])
    return retrieved_contexts, query_embedding

prompt generation for ENFP

In [ ]:
def generate_design_prompts_enfp(k=3):
    prompts = []
    top_k_references_all_enfp = []  # store top-k references per prompt

    for i, context in enumerate(designs):
        query_embedding = np.expand_dims(embeddings[i], axis=0)
        distances, indices = faiss_index.search(query_embedding, k + 1)
        similar_indices = [idx for idx in indices[0] if idx != i][:k]

        similar_contexts = [designs[j] for j in similar_indices]

        prompt = f"""You are a visionary interior designer with a deep understanding of personality-driven design. Your task is to create a space that not only represents the user’s personality but also nurtures their emotional needs.

**Primary Design Inspiration:**
{context}

**Related Inspirations:**
""" + "\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)])

        prompt += f"""
Design this space thoughtfully, considering the user’s emotional connection to the environment:
- Choose a **primary color** that captures the user’s mood and enhances their personality.
- Select **furniture** that aligns with their emotional needs, comfort, and expression.
- Pick **lighting** that either energizes or calms, depending on their preference.

Finally, describe the **overall emotional vibe** of the space and explain how the elements of design (color, furniture, and lighting) seamlessly come together to create a cohesive and emotionally fulfilling environment.
"""

        prompts.append(prompt)
        top_k_references_all_enfp.append(similar_contexts)  # save top-k references

    return prompts, top_k_references_all_enfp

design_prompts_enfp, top_k_references_all_enfp = generate_design_prompts_enfp(k=3)

# Preview
for p_enfp, refs_enfp in zip(design_prompts_enfp[:1], top_k_references_all_enfp[:1]):
    print("Prompt:\n", p_enfp)
    print("\nTop-k References:\n", refs_enfp)
    print("="*80)



In [ ]:
import requests
import time


api_key="myapi_key";
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
#changed the api and version from llama3 70 b to llama 3.3 70b versatile
def generate_llama3_recommendations(prompts, model="llama-3.3-70b-versatile", temperature=0.5):
    responses = []
    for i, prompt in enumerate(prompts):
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a brilliant, emotionally intuitive interior designer. Make your recommendations creative,concise, expressive,complete and glitterified."},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "max_tokens": 768
        }

        try:
            res = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)

            # Check for errors
            if res.status_code != 200:
                print(f"[❌] Prompt {i+1} failed: HTTP {res.status_code} → {res.text}")
                responses.append("")
                continue

            data = res.json()
            output = data["choices"][0]["message"]["content"]
            print(f"[✅] Prompt {i+1} completed.")
            responses.append(output)

        except requests.exceptions.RequestException as e:
            print(f"[❌] Network error for prompt {i+1}: {e}")
            responses.append("")
        except Exception as e:
            print(f"[❌] Unexpected error for prompt {i+1}: {e}")
            responses.append("")

        time.sleep(0.5)  # avoid rate limits
    return responses



In [ ]:
# Assuming you've already created semantic_prompts
llama3_outputs_enfp = generate_llama3_recommendations(design_prompts_enfp[:1])  # Do first
display(llama3_outputs_enfp)

In [ ]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
import numpy as np
from nltk.util import ngrams

# --- Load SBERT ---
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

# --- Text normalization ---
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def semantic_p_bleu(hypo_text, topk_refs, n=3, sim_threshold=0.5):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    scores = []
    for ngram_size in range(1, n+1):
        hypo_ngrams = list(ngrams(hypo_tokens, ngram_size))
        hypo_str_ngrams = [" ".join(ng) for ng in hypo_ngrams]
        hypo_emb = model_sbert.encode(hypo_str_ngrams, convert_to_tensor=True)

        max_match_scores = []
        for ref_text in topk_refs:
            ref_tokens = normalize_text(ref_text).split()
            ref_ngrams = list(ngrams(ref_tokens, ngram_size))
            ref_str_ngrams = [" ".join(ng) for ng in ref_ngrams]
            if not ref_str_ngrams:
                continue
            ref_emb = model_sbert.encode(ref_str_ngrams, convert_to_tensor=True)

            cos_sim = util.cos_sim(hypo_emb, ref_emb)
            max_per_hypo = cos_sim.max(dim=1).values
            max_per_hypo = (max_per_hypo >= sim_threshold).float()
            max_match_scores.append(max_per_hypo.sum().item() / len(hypo_ngrams))
        scores.append(max(max_match_scores) if max_match_scores else 0)
    return np.mean(scores)

# --- Semantic ROUGE-L (reference-free) ---
def semantic_rouge_l(hypo_text, topk_refs, sim_threshold=0.6):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    max_score = 0.0
    hypo_emb = model_sbert.encode(hypo_tokens, convert_to_tensor=True)

    for ref_text in topk_refs:
        ref_tokens = normalize_text(ref_text).split()
        if not ref_tokens:
            continue
        ref_emb = model_sbert.encode(ref_tokens, convert_to_tensor=True)
        cos_sim = util.cos_sim(hypo_emb, ref_emb)
        binary_matches = (cos_sim >= sim_threshold).float()
        precision = binary_matches.max(dim=1).values.mean().item()
        recall = binary_matches.max(dim=0).values.mean().item()
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        max_score = max(max_score, f1)
    return max_score

# --- Evaluation ---
results = []

for i, hypo_text in enumerate(llama3_outputs_enfp):
    topk_refs_enfp = top_k_references_all_enfp[i]  # exact 3 similar inputs used in prompt

    p_bleu = semantic_p_bleu(hypo_text, topk_refs_enfp)
    rouge_l = semantic_rouge_l(hypo_text, topk_refs_enfp)

    # Compute BERTScore separately

    results.append({
        "P-BLEU": round(p_bleu,3),
        "ROUGE-L": round(rouge_l,3),
    })

df = pd.DataFrame(results)
display(df)


In [ ]:


# --- Load SBERT model ---
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

# --- Initialize results list ---
results = []

# Loop through all generated recommendations
for i in range(len(llama3_outputs_enfp)):
    generated_enfp = llama3_outputs_enfp[i]
    refs_enfp = top_k_references_all_enfp[i]  # list of top-k reference strings for this prompt

    # --- BERTScore (precision, recall, F1) ---
    # Pick best reference based on max SBERT similarity for BERTScore
    results_enfp = max(refs_enfp, key=lambda r: util.cos_sim(
        model_sbert.encode(r, convert_to_tensor=True),
        model_sbert.encode(generated_enfp, convert_to_tensor=True)
    ).item())
    P, R, F1 = bert_score([generated_enfp], [results_enfp], lang="en")
    P, R, F1 = float(P[0]), float(R[0]), float(F1[0])

    # --- SBERT semantic alignment (max cosine similarity among top-k refs) ---
    emb_refs = model_sbert.encode(refs_enfp, convert_to_tensor=True)
    emb_gen = model_sbert.encode(generated_enfp, convert_to_tensor=True)
    cos_scores = util.cos_sim(emb_gen, emb_refs)
    sbert_alignment = cos_scores.max().item()

    # --- Save results ---
    results.append({
        "BERTScore_P": round(P, 3),
        "BERTScore_R": round(R, 3),
        "BERTScore_F1": round(F1, 3),
        "SBERT_Alignment": round(sbert_alignment, 3)
    })

# --- Create DataFrame ---
df_metrics = pd.DataFrame(results)
display(df_metrics)

# ==========================================================
# --- Graphical Representation ---
# ==========================================================
plt.figure(figsize=(12,6))
plt.plot(df_metrics.index, df_metrics["BERTScore_F1"], marker='o', label="BERTScore F1")
plt.plot(df_metrics.index, df_metrics["SBERT_Alignment"], marker='s', label="SBERT Alignment (Cosine)")
plt.title("Evaluation of Generated Recommendations")
plt.xlabel("Prompt Index")
plt.ylabel("Score")
plt.ylim(0, 1.0)
plt.grid(True)
plt.legend()
plt.show()

# Optional: side-by-side bar chart
df_metrics[["BERTScore_F1","SBERT_Alignment"]].plot(
    kind="bar", figsize=(12,6), rot=0, title="BERTScore vs SBERT Alignment"
)
plt.ylabel("Score")
plt.show()


# Prompt Generation

In [ ]:
def generate_semantic_prompts(k=3):
    prompts = []
    for i, context in enumerate(designs):
        query_embedding = np.expand_dims(embeddings[i], axis=0)
        distances, indices = faiss_index.search(query_embedding, k + 1)  # +1 to skip self
        similar_indices = [idx for idx in indices[0] if idx != i][:k]

        similar_contexts = [designs[j] for j in similar_indices]

        # New RAG-optimized, richer prompt
        prompt = f"""You are a visionary interior designer specializing in crafting dynamic, expressive spaces for high-energy personalities.

Your main design reference is:
{context}

You have also consulted the following related design inspirations:
""" + "\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)]) + """

Analyze these references carefully. Then, creatively blend their dominant color trends, furniture ideas, and emotional vibes into a new, cohesive design recommendation.

In your response:
- Start by summarizing the common color themes and emotional tones found across all references.
- Then propose a bold new dominant color palette and explain why it captures adventurous energy.
- Suggest 2–3 signature furniture pieces that match the energetic and free-spirited personality.
- Evoke vivid emotions (like excitement, freedom, boldness) as if painting a living picture of the space.

 Think of yourself not just as a designer, but as an artist capturing the soul of a free-spirited adventurer.
"""

        prompts.append(prompt)
    return prompts

# --- Now generate the prompts ---
semantic_prompts = generate_semantic_prompts(k=3)

# --- Preview first few prompts ---
for p in semantic_prompts[:5]:
    print(p)
    print("="*80)


# Prompt Saving for fine tunning instruction and input

In [ ]:
import numpy as np
import json
import random

def generate_semantic_prompts(designs, embeddings, faiss_index, k=5, max_prompts=50):
    prompts = []
    used_indices = set()

    design_indices = list(range(len(designs)))
    random.shuffle(design_indices)  # Add randomness

    for i in design_indices:
        if i in used_indices:
            continue

        query_embedding = np.expand_dims(embeddings[i], axis=0)
        distances, indices = faiss_index.search(query_embedding, k + 1)

        # Filter out self and already used indices
        similar_indices = [idx for idx in indices[0] if idx != i and idx not in used_indices][:k]

        if len(similar_indices) < k:
            continue  # Not enough valid neighbors

        similar_contexts = [designs[j] for j in similar_indices]

        # Create prompt
        inspiration_block = "\n".join([f"{j+1}. {text}" for j, text in enumerate(similar_contexts)])

        prompt_text = f"""You are a visionary interior designer specializing in crafting expressive, high-energy spaces.

Below are 5 real-world design inspirations for adventurous and free-spirited individuals:
{inspiration_block}

Analyze all inspirations carefully. Then craft a bold new interior design concept that merges dominant color themes, furniture moods, and emotional expressions from above.

Your response should:
- Summarize color/emotion trends seen across all examples
- Propose a new, energetic dominant color palette with justification
- Recommend 2–3 bold furniture ideas that embody freedom & motion
- Convey a vivid emotional atmosphere through storytelling

Design like an artist who paints emotion into every space."""

        prompts.append({
            "instruction": "Generate a bold, expressive interior design concept from 5 inspirational references.",
            "input": prompt_text.strip()
        })

        used_indices.update(similar_indices + [i])

        if len(prompts) >= max_prompts:
            break  # Limit reached

    return prompts


# === Run it ===
# Make sure `designs`, `embeddings`, and `faiss_index` are defined earlier in your notebook/script

semantic_prompt_records = generate_semantic_prompts(designs, embeddings, faiss_index, k=5, max_prompts=50)

output_path = "semantic_instruction_input_50.json"
with open(output_path, "w") as f:
    json.dump(semantic_prompt_records, f, indent=2)

print(f"[✅] {len(semantic_prompt_records)} prompts saved to {output_path}")


In [ ]:
import json

def validate_json(filepath):
    try:
        with open(filepath, "r") as f:
            json.load(f)
        print("✅ JSON is valid.")
    except json.JSONDecodeError as e:
        print(f"❌ JSONDecodeError: {e}")
        with open(filepath, "r") as f:
            lines = f.readlines()
        print("\nLast few lines before failure:")
        for line in lines[max(0, e.lineno - 3): e.lineno + 2]:
            print(line.strip())

validate_json("semantic_instruction_input_50.json")


# Generate Output for fine tunning lama 3

In [ ]:
import requests
import time
import json

# === Config ===
api_key = "mykey"  # Replace this
model = "llama3-70b-8192"
temperature = 0.5
input_file = "semantic_instruction_input_50.json"
output_file = "semantic_instruction_input_with_output.json"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# === Load dataset ===
with open(input_file, "r") as f:
    dataset = json.load(f)

# === Process prompts ===
for i, item in enumerate(dataset):
    # ✅ Skip if already has valid output
    if "output" in item and item["output"].strip():
        print(f"[⏭️] Prompt {i+1} already completed. Skipping.")
        continue

    prompt = item["input"]

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a brilliant, emotionally intuitive interior designer. Make your recommendations creative, expressive, and glitterified."},
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 512
    }

    success = False
    retries = 3
    while not success and retries > 0:
        try:
            response = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
            data = response.json()

            if "choices" in data:
                output = data["choices"][0]["message"]["content"]
                dataset[i]["output"] = output.strip()
                print(f"[✅] Prompt {i+1} completed.")
                success = True
            elif "error" in data and "rate_limit" in data["error"].get("code", ""):
                wait_time = 5
                print(f"[⏳] Rate limit hit. Waiting {wait_time}s before retrying...")
                time.sleep(wait_time)
                retries -= 1
            else:
                print(f"[❌] Prompt {i+1} failed: Unexpected response. {data}")
                dataset[i]["output"] = ""
                break

        except Exception as e:
            print(f"[❌] Prompt {i+1} error: {e}")
            dataset[i]["output"] = ""
            break

    # ✅ Save progress after each prompt (crash-safe)
    with open(output_file, "w") as f:
        json.dump(dataset, f, indent=2)

    time.sleep(1.2)  # Gentle throttle to reduce risk of limit

print("\n[🏁] All prompts processed.")


In [ ]:
import requests
import time
import json

# === Config ===
api_key = "mykey"  # Replace with your actual API key
model = "llama3-70b-8192"
temperature = 0.5
input_file = "semantic_instruction_input_with_output.json"
output_file = "semantic_instruction_input_with_output.json"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# === Load dataset ===
with open(input_file, "r") as f:
    dataset = json.load(f)

# === Resume from failed outputs ===
for i, item in enumerate(dataset):
    if "output" in item and item["output"].strip():
        print(f"[⏭️] Prompt {i+1} already completed. Skipping.")
        continue

    prompt = item["input"]

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a brilliant, emotionally intuitive interior designer. Make your recommendations creative, expressive, and glitterified."},
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 512
    }

    retries = 5
    while retries > 0:
        try:
            response = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
            data = response.json()

            if "choices" in data:
                output = data["choices"][0]["message"]["content"]
                dataset[i]["output"] = output.strip()
                print(f"[✅] Prompt {i+1} completed.")
                break

            elif "error" in data:
                err_msg = data["error"]["message"].lower()
                if "service unavailable" in err_msg or "internal_server_error" in err_msg:
                    print(f"[🛑] Groq service unavailable. Retrying prompt {i+1} in 10s...")
                    time.sleep(10)
                    retries -= 1
                else:
                    print(f"[❌] Prompt {i+1} failed: {data}")
                    dataset[i]["output"] = ""
                    break
            else:
                print(f"[❌] Prompt {i+1} unexpected format: {data}")
                dataset[i]["output"] = ""
                break

        except Exception as e:
            print(f"[❌] Prompt {i+1} crashed: {e}")
            retries -= 1
            time.sleep(5)

    # ✅ Save after every prompt
    with open(output_file, "w") as f:
        json.dump(dataset, f, indent=2)

    time.sleep(1.2)  # Slow down to reduce chance of hitting limits

print("\n[🏁] Recovery run complete.")


# Evalution of generated recommendations

In [ ]:
!pip install nltk rouge-score bert-score
import nltk
nltk.download('punkt')


In [ ]:
import json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import nltk
nltk.download("punkt")

# === Load dataset ===
with open("semantic_instruction_input_with_output_completed.json", "r") as f:
    dataset = json.load(f)

# === Evaluators ===
bleu_smooth = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# === Evaluate ===
results = []

print("\n=== BLEU and ROUGE Evaluation ===\n")

for i, item in enumerate(dataset):
    prompt_text = (item.get("instruction", "") + " " + item.get("input", "")).strip()
    output_text = item.get("output", "").strip()

    if not output_text:
        print(f"[⚠️] Skipping Prompt {i+1} – output missing.")
        continue

    # Tokenize for BLEU
    ref_tokens = nltk.word_tokenize(prompt_text)
    hyp_tokens = nltk.word_tokenize(output_text)

    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=bleu_smooth)
    rouge_l = rouge.score(prompt_text, output_text)["rougeL"].fmeasure

    # Append scores
    item["bleu"] = round(bleu, 4)
    item["rougeL"] = round(rouge_l, 4)

    results.append((i + 1, item["bleu"], item["rougeL"]))

    print(f"Prompt {i+1}:")
    print(f"  BLEU:    {item['bleu']}")
    print(f"  ROUGE-L: {item['rougeL']}\n")

# === Save with scores ===
with open("semantic_instruction_with_scores.json", "w") as f:
    json.dump(dataset, f, indent=2)

print("\n✅ Evaluation complete. Results saved to 'semantic_instruction_with_scores.json'")


In [ ]:
import requests
import time

api_key = "mykey"  # Replace with your actual key

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

def generate_llama3_recommendations(prompts, model="llama3-70b-8192", temperature=0.5):
    responses = []
    for i, prompt in enumerate(prompts):
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a brilliant, emotionally intuitive interior designer. Make your recommendations creative, expressive, and glitterified."},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "max_tokens": 512
        }

        try:
            res = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
            output = res.json()["choices"][0]["message"]["content"]
            print(f"[✅] Prompt {i+1} completed.")
            responses.append(output)
        except Exception as e:
            print(f"[❌] Prompt {i+1} failed: {e}")
            responses.append("")

        time.sleep(0.5)  # avoid rate limits
    return responses


In [ ]:
# Assuming you've already created semantic_prompts
llama3_outputs = generate_llama3_recommendations(semantic_prompts[:5])  # Do first


In [ ]:
for i, out in enumerate(llama3_outputs):
    print(f"\n Prompt {i+1} Recommendation:\n{out}")
    print("=" * 100)


# DATA Prepration for fine tunning

In [ ]:
import numpy as np
import json
import requests
import time
import os

api_key = "my_key"
output_json_path = "finetune_llama3_dataset.json"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# Ensure output file exists
if not os.path.exists(output_json_path):
    with open(output_json_path, "w") as f:
        f.write("[")

def generate_semantic_prompt(i, k=5):
    context = designs[i]
    query_embedding = np.expand_dims(embeddings[i], axis=0)
    distances, indices = faiss_index.search(query_embedding, k + 1)
    similar_indices = [idx for idx in indices[0] if idx != i][:k]
    similar_contexts = [designs[j] for j in similar_indices]

    prompt = f"""You are a visionary interior designer specializing in crafting dynamic, expressive spaces for high-energy personalities.

Your main design reference is:
{context}

You have also consulted the following related design inspirations:
""" + "\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)]) + """

Analyze these references carefully. Then, creatively blend their dominant color trends, furniture ideas, and emotional vibes into a new, cohesive design recommendation.

In your response:
- Start by summarizing the common color themes and emotional tones found across all references.
- Then propose a bold new dominant color palette and explain why it captures adventurous energy.
- Suggest 2–3 signature furniture pieces that match the energetic and free-spirited personality.
- Evoke vivid emotions (like excitement, freedom, boldness) as if painting a living picture of the space.

Think of yourself not just as a designer, but as an artist capturing the soul of a free-spirited adventurer.
"""
    return prompt

def call_groq_api(prompt, model="llama3-70b-8192", temperature=0.5):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a brilliant, emotionally intuitive interior designer. Make your recommendations creative, expressive, and glitterified."},
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 512
    }

    try:
        response = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
        output = response.json()["choices"][0]["message"]["content"]
        return output
    except Exception as e:
        print(f"[❌] API failed: {e}")
        return None

# Loop through designs, generate, write one at a time
total = len(designs)
for i in range(total):
    prompt = generate_semantic_prompt(i, k=5)
    response = call_groq_api(prompt)

    if response:
        item = {
            "instruction": "Design an emotionally expressive interior space based on inspirations.",
            "input": prompt.strip(),
            "output": response.strip()
        }

        # Write to file incrementally
        with open(output_json_path, "a") as f:
            json.dump(item, f)
            if i < total - 1:
                f.write(",\n")  # add comma between records
            else:
                f.write("\n")   # last record

        print(f"[✅] Prompt {i+1} saved.")
    else:
        print(f"[⚠️] Skipped Prompt {i+1} due to API failure.")

    time.sleep(0.5)  # prevent rate limits

# Properly close JSON array if it ended
with open(output_json_path, "a") as f:
    f.write("]")


# Converting Recommendation → Visual Prompt for Diffusion

In [ ]:
def extract_visual_prompt_from_recommendation(recommendation):
    # Lightweight approach — you could use GPT to automate this instead
    import re
    palette_match = re.findall(r"bold new.*palette.*?:\s*(.+?)(\.|\n)", recommendation, re.IGNORECASE)
    colors = palette_match[0][0] if palette_match else "vibrant earth tones"

    furniture_match = re.findall(r"(?:furniture pieces.*?:|include:)\s*(.+?)(\.|\n)", recommendation, re.IGNORECASE)
    furniture = furniture_match[0][0] if furniture_match else "a velvet sofa and mid-century coffee table"

    return f"A dynamic interior with {colors}, featuring {furniture}, natural lighting, and expressive décor."


# Evaluation

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from nltk.util import ngrams
from nltk import FreqDist
!pip install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
!pip install bert-score  # Ensure BERTScore library is installed

from bert_score import score

# --- Functions ---

# Relevance Score (Cosine Similarity between prompt and recommendation)
def relevance_score(prompt, recommendation, model):
    prompt_vec = model.encode([prompt])  # Use appropriate model to encode text
    rec_vec = model.encode([recommendation])  # Same for recommendation
    return float(cosine_similarity(prompt_vec, rec_vec)[0][0])

# Creativity Score (Semantic Diversity based on bigrams)
def creativity_score(recommendation):
    n = 2  # Bigrams
    recommendation_ngrams = list(ngrams(recommendation.split(), n))
    bigram_freq = FreqDist(recommendation_ngrams)
    return len(bigram_freq)

# Sentiment Score (Positive/Negative feeling)
def sentiment_score(text):
    analyzer = SentimentIntensityAnalyzer()
    sentiment = analyzer.polarity_scores(text)
    return sentiment['compound']

# BERTScore Function
def bertscore(prompt, recommendation):
    P, R, F1 = score([recommendation], [prompt], lang="en", model_type="microsoft/deberta-xlarge-mnli", verbose=False)
    return float(F1[0])  # Return F1 Score

def bert_precision(prompt, recommendation):
    P, R, F1 = score([recommendation], [prompt], lang="en", model_type="microsoft/deberta-xlarge-mnli", verbose=False)
    return float(P[0])  # Return Precision Score


# --- Evaluation Loop ---

results = []

for i in range(5):  # Assuming you have 10 prompts and 10 outputs
    used_prompt = semantic_prompts[i]
    generated_recommendation = llama3_outputs[i]

    row = {
        "Prompt": used_prompt[:50].strip().replace("\n", " ") + "...",  # Shorten for readability
        "Recommendation": generated_recommendation[:100].strip().replace("\n", " ") + "...",
        "Relevance Score": round(relevance_score(used_prompt, generated_recommendation, model), 3),
        "Creativity Score": round(creativity_score(generated_recommendation), 2),
        "Sentiment": round(sentiment_score(generated_recommendation), 3),
#        "BERTScore (F1)": round(bertscore(used_prompt, generated_recommendation), 3),
#        "BERTScore (Precision)": round(bert_precision(used_prompt, generated_recommendation), 3),
#        "BERTScore (Recall)": round(bert_recall(used_prompt, generated_recommendation), 3),
    }
    results.append(row)

# --- Display Results ---

df = pd.DataFrame(results)
display(df)


In [ ]:
# Sort by Relevance first, then Creativity
best_recommendation = df.sort_values(by=["Relevance Score", "Creativity Score"], ascending=[False, False]).iloc[0]
display(best_recommendation)

### Apply GPT 4-0-mini

In [ ]:
pip install openai


In [ ]:
from openai import OpenAI
import time

# Initialize OpenAI client
client = OpenAI(api_key="MY_KEy")  # Replace with your key
ans = client.models.list()

for model in ans.data:
    print(model.id)
# Function to get GPT responses for your auto-generated prompts
def generate_responses_from_prompts(prompts, model="gpt-4o-mini", temperature=0.7, max_tokens=200):
    responses = []

    for i, prompt in enumerate(prompts):
        print(f"Sending prompt {i+1}/{len(prompts)}...")

        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a helpful interior design assistant."},
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )
            response_text = completion.choices[0].message.content.strip()
            responses.append((prompt, response_text))
        except Exception as e:
            print(f"Error with prompt {i}: {e}")
            responses.append((prompt, "ERROR"))

        time.sleep(1)  # Avoid rate limits

    return responses


In [ ]:
# Use your generated prompts (e.g., from generate_automatic_prompts())
test_prompts = semantic_prompts[:5] # Only send first 5 to avoid cost/rate issues

results = generate_responses_from_prompts(test_prompts)

# Print results
for i, (prompt, response) in enumerate(results):
    print(f"\n--- PROMPT {i+1} ---\n{prompt}")
    print(f"\n--- GPT-4o-mini RESPONSE ---\n{response}")
    print("="*80)


In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from nltk.util import ngrams
from nltk import FreqDist
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer

# --- Load the Sentence Transformer Model ---
model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight + fast + good performance

# --- Scoring Functions ---

def relevance_score(prompt, recommendation, model):
    prompt_vec = model.encode([prompt])
    rec_vec = model.encode([recommendation])
    return float(cosine_similarity(prompt_vec, rec_vec)[0][0])

def creativity_score(recommendation):
    n = 2  # Using bigrams
    recommendation_ngrams = list(ngrams(recommendation.split(), n))
    bigram_freq = FreqDist(recommendation_ngrams)
    return len(bigram_freq)

def sentiment_score(text):
    analyzer = SentimentIntensityAnalyzer()
    sentiment = analyzer.polarity_scores(text)
    return sentiment['compound']


# --- Evaluation Section ---

results = []

# Assume: semantic_prompts is a list of design prompts
# Assume: generate_responses_from_prompts(prompts) returns a list of generated recommendations

test_prompts = semantic_prompts[:5]  # Evaluate first 5 prompts only (safe for cost/rate)
generated_responses = generate_responses_from_prompts(test_prompts)  # List of responses

for prompt, response in zip(test_prompts, generated_responses):
    row = {
        "Prompt": prompt[:80].strip().replace("\n", " ") + "...",  # Truncate for display
        "Recommendation": response[:120].strip().replace("\n", " ") + "...",
        "Relevance Score": round(relevance_score(prompt, response, model), 3),
        "Creativity Score": round(creativity_score(response), 2),
        "Sentiment": round(sentiment_score(response), 3),
    }
    results.append(row)

# --- Display Results ---

df_evaluation = pd.DataFrame(results)
display(df_evaluation)




# Applied RAG on Without Random over sampling and except Description

In [ ]:
df_type = pd.read_csv(f"{user_input}_designs.csv")



# --- Optional: Show column names to verify structure ---
print("✅ Columns in loaded CSV:", df_type.columns.tolist())


# --- Construct context using all relevant features ---
df_type["design_context_without_desc"] = df_type.apply(
    lambda row: f"""
    Personality Type: {row['personality_type']}
    Design Details:
    - Dominant Color: {row['dominant_color']}
    - Color Palette: {row['palette']}
    - style: {row['predicted_style']}
    """,
    axis=1
)

# --- Convert context column to list for downstream processing ---
designs = df_type["design_context_without_desc"].tolist()

# --- Display first few design contexts ---
print("\n✅ Sample design contexts:")
for d in designs[:3]:
    print(d)
    print("=" * 50)

In [ ]:
import ast
import matplotlib.colors as mcolors

# Parse the palette string into a list of RGB tuples
def parse_palette(palette_str):
    try:
        return ast.literal_eval(palette_str)
    except:
        return []

# Convert RGB to hex color string
def rgb_to_color_name(rgb_tuple):
    try:
        norm_rgb = tuple(x / 255.0 for x in rgb_tuple)
        return mcolors.to_hex(norm_rgb)
    except:
        return "unknown"

# Apply processing to the DataFrame
df_type['palette_names'] = df_type['palette'].apply(
    lambda x: ' '.join([rgb_to_color_name(rgb) for rgb in parse_palette(x)])
)

# Show example output
print(df_type[['palette', 'palette_names']].head())


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle
import faiss

# Initialize model
model1 = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# Combine all relevant features into one embedding string
embedding_inputs = [
    f"Personality: {row['personality_type']} | Color: {row['dominant_color']} | Style: {row['predicted_style']} | Palette: {row['palette_names']}"
    for _, row in df_type.iterrows()
]

# Generate embeddings
embeddings = model1.encode(embedding_inputs)
embeddings = np.array(embeddings).astype('float32')

# Save embeddings and context
with open('design_embeddings.pkl', 'wb') as f:
    pickle.dump({
        'embeddings': embeddings,
        'design_contexts': embedding_inputs  # full string input used
    }, f)

# Save FAISS index
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)
faiss.write_index(faiss_index, 'design_index.faiss')

print("✅ Embeddings generated and saved with personality type included.")


In [ ]:
# Load embeddings
with open('design_embeddings.pkl', 'rb') as f:
    data = pickle.load(f)
    embeddings = data['embeddings']
    designs = data['design_contexts']

# Load FAISS index
index = faiss.read_index('design_index.faiss')

# Verify loading
print(f"Loaded {len(designs)} designs with {embeddings.shape[1]}-dim embeddings")

In [ ]:
# Create FAISS index and add embeddings
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)  # <-- THIS is crucial

# Save index
faiss.write_index(index, 'design_index.faiss')


In [ ]:
def retrieve_designs(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)
    retrieved_contexts = "\n\n".join([designs[i] for i in indices[0]])
    return retrieved_contexts, query_embedding

In [ ]:
def generate_semantic_prompts(k=3):
    prompts = []
    for i, context in enumerate(designs):
        query_embedding = np.expand_dims(embeddings[i], axis=0)
        distances, indices = faiss_index.search(query_embedding, k + 1)  # +1 to skip self
        similar_indices = [idx for idx in indices[0] if idx != i][:k]

        similar_contexts = [designs[j] for j in similar_indices]

        # Expressive prompt for generation
        prompt = f"""You are an expert interior designer.

Generate a creative and emotionally resonant design recommendation based on:
{context}

Here are {k} semantically similar design contexts for inspiration:
""" + "\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)])

        prompts.append(prompt)
    return prompts

# Generate prompts for the first few rows (e.g., for testing)
semantic_prompts1 = generate_semantic_prompts(k=3)

# Preview first 3 prompts
for p in semantic_prompts1[:3]:
    print(p)
    print("="*80)

# Only prompt generation

In [ ]:
def generate_one_semantic_prompt(k=3):
    i = 0  # use the first design row
    context = designs[i]
    query_embedding = np.expand_dims(embeddings[i], axis=0)
    distances, indices = faiss_index.search(query_embedding, k + 1)

    similar_indices = [idx for idx in indices[0] if idx != i][:k]
    similar_contexts = [designs[j] for j in similar_indices]

    # Create a single expressive prompt
    prompt = f"""You are an expert interior designer.

Generate a creative and emotionally resonant design recommendation based on:
{context}

Here are {k} semantically similar design contexts for inspiration:
""" + "\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)])

    return prompt


In [ ]:
# Generate one optimized semantic prompt
optimal_prompt = generate_one_semantic_prompt(k=3)

# Print it to verify
print(optimal_prompt)


# Generate Recommendations llama3-8b-8192 , llama3-70b-8192  

In [ ]:
import requests
import time

api_key = "mykey"  # Replace with your actual key

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

def generate_llama3_recommendations(prompts, model="llama3-70b-8192", temperature=0.8):
    responses = []
    for i, prompt in enumerate(prompts):
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a brilliant, emotionally intuitive interior designer. Make your recommendations creative, expressive, and glitterified."},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "max_tokens": 512
        }

        try:
            res = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
            output = res.json()["choices"][0]["message"]["content"]
            print(f"[✅] Prompt {i+1} completed.")
            responses.append(output)
        except Exception as e:
            print(f"[❌] Prompt {i+1} failed: {e}")
            responses.append("")

        time.sleep(0.5)  # avoid rate limits
    return responses


In [ ]:
# Assuming you've already created semantic_prompts
llama3_outputs_ = generate_llama3_recommendations(semantic_prompts1[:1])  # Do first 5 for testing


In [ ]:
for i, out in enumerate(llama3_outputs_):
    print(f"\n Prompt {i+1} Recommendation:\n{out}")
    print("=" * 100)


# Evalution matrix

In [ ]:
!pip install vaderSentiment


In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from nltk.util import ngrams
from nltk import FreqDist
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Relevance Score (Cosine Similarity between prompt and recommendation)
def relevance_score(prompt, recommendation):
    prompt_vec = model.encode([prompt])
    rec_vec = model.encode([recommendation])
    return float(cosine_similarity(prompt_vec, rec_vec)[0][0])

# Creativity Score (Using semantic diversity)
def creativity_score(recommendation):
    n = 2  # Bigrams
    recommendation_ngrams = list(ngrams(recommendation.split(), n))
    bigram_freq = FreqDist(recommendation_ngrams)
    return len(bigram_freq)

# Sentiment Score (Using VADER Sentiment)
def sentiment_score(text):
    analyzer = SentimentIntensityAnalyzer()
    sentiment = analyzer.polarity_scores(text)
    return sentiment['compound']

# Evaluate the recommendations — match each prompt with its corresponding recommendation
results = []

for i, (prompt_text, recommendation) in enumerate(zip(semantic_prompts1[:1], llama3_outputs_)):
    row = {
        "Prompt": prompt_text[:50].strip().replace("\n", " ") + "...",
        "Recommendation": recommendation[:200].strip().replace("\n", " ") + "...",
        "Relevance Score": round(relevance_score(prompt_text, recommendation), 2),
        "Creativity Score": round(creativity_score(recommendation), 2),
        "Sentiment": round(sentiment_score(recommendation), 2),
    }
    results.append(row)

# Show results
df = pd.DataFrame(results)
display(df)


# Recommendations using llama3-8b

In [ ]:
import requests
import time

api_key = "mykey"  # Replace with your actual key

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

def generate_llama3_recommendations(prompts, model="llama3-8b-8192", temperature=0.8):
    responses = []
    for i, prompt in enumerate(prompts):
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a brilliant, emotionally intuitive interior designer. Make your recommendations creative, expressive, and glitterified."},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "max_tokens": 512
        }

        try:
            res = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)
            output = res.json()["choices"][0]["message"]["content"]
            print(f"[✅] Prompt {i+1} completed.")
            responses.append(output)
        except Exception as e:
            print(f"[❌] Prompt {i+1} failed: {e}")
            responses.append("")

        time.sleep(0.5)  # avoid rate limits
    return responses


In [ ]:
llama3_outputs_8b = generate_llama3_recommendations(semantic_prompts[:1])  # Do first 5 for testing


In [ ]:
for i, out in enumerate(llama3_outputs_8b):
    print(f"\n Prompt {i+1} Recommendation:\n{out}")
    print("=" * 100)


In [ ]:
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
from nltk.util import ngrams
from nltk import FreqDist
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Relevance Score (Cosine Similarity between prompt and recommendation)
def relevance_score(prompt, recommendation):
    prompt_vec = model.encode([prompt])
    rec_vec = model.encode([recommendation])
    return float(cosine_similarity(prompt_vec, rec_vec)[0][0])

# Creativity Score (Using semantic diversity)
def creativity_score(recommendation):
    # Compute n-grams and calculate diversity
    n = 2  # We will use bigrams (you can adjust this for higher n-grams)
    recommendation_ngrams = list(ngrams(recommendation.split(), n))
    bigram_freq = FreqDist(recommendation_ngrams)
    return len(bigram_freq)  # Number of unique bigrams, higher = more diverse

# Sentiment Score (Using VADER Sentiment)
def sentiment_score(text):
    analyzer = SentimentIntensityAnalyzer()
    sentiment = analyzer.polarity_scores(text)
    return sentiment['compound']  # Returns a score between -1 and 1



# Generate the optimal prompt using the function
optimal_prompt = generate_one_semantic_prompt(k=3)

# Get the recommendations from LLaMA 3 model (test with the first prompt)
llama3_outputs_8b = generate_llama3_recommendations([optimal_prompt])

# Evaluate the recommendations and store results in a list of dictionaries
results = []

for i, recommendation in enumerate(llama3_outputs):
    row = {
        "Prompt": optimal_prompt[:50] + "...",  # Show first 50 characters for brevity
        "Recommendation": recommendation[:50] + "...",  # First 50 characters of the recommendation
        "Relevance Score": round(relevance_score(optimal_prompt, recommendation), 2),
        "Creativity Score": round(creativity_score(recommendation), 2),
        "Sentiment": round(sentiment_score(recommendation), 2),  # Sentiment polarity between -1 and 1
    }
    results.append(row)

# Convert to DataFrame and print the evaluation matrix
df = pd.DataFrame(results)
display(df)
